# Decoder probe: is the latent blind, or is the decoder just undertrained?

The M3 world model (`ckpt_final.pt`) reconstructs background biome color fine but
drops HUD digits, mobs, trees, and precise player pose almost entirely. That's
consistent with two very different root causes that look identical in the recon
panel:

1. **The `[h,z]` latent doesn't encode this content at all** (encoder/RSSM
   problem, expensive to fix -- more capacity, different KL budget, etc.)
2. **The latent has the info, but the jointly-trained decoder never learned to
   render it** -- it was competing the whole time against reward/continue/KL
   gradients for the same optimizer steps (cheap to fix -- just train a decoder
   harder/longer on the frozen features).

This notebook tells them apart with a standard **probe-decodability test**:
freeze the trained encoder + RSSM, cache their `[h,z]` features once for a few
thousand real frames, then train a **brand-new decoder from scratch** purely on
those frozen features -- full attention, no competing losses, cheap per-step
(no encoder/RSSM forward-backward needed once cached).

- If the probe decoder recovers mobs/trees/HUD digits that the original
  decoder missed -> the information was there all along. Fix: give the decoder
  more dedicated training (e.g. a decoder-only fine-tuning pass), not touch
  the RSSM/encoder/loss balance at all.
- If the probe decoder is just as blind -> the RSSM/encoder genuinely isn't
  encoding that content. That's a real, more expensive problem worth knowing
  about *before* M4, since M4's imagined rollouts depend on the same latent.

Requires `checkpoints/m3_world_model/ckpt_final.pt` to exist (run
`05_train_world_model.ipynb` first).

In [ ]:
from __future__ import annotations

import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from IPython.display import clear_output, display

from models.decoder import Decoder
from models.heads import rssm_features
from models.preprocess import nchw_float_to_nhwc_uint8, nhwc_uint8_to_nchw_float
from models.rssm import one_hot_action
from models.world_model import WorldModel
from training.device import get_device
from training.losses import gradient_l1_loss
from training.replay_buffer import ReplayBuffer

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
%matplotlib inline

CONFIG = Path("configs/m3_world_model.yaml")
CHECKPOINT = Path("checkpoints/m3_world_model/ckpt_final.pt")

with CONFIG.open() as f:
    cfg = yaml.safe_load(f)
device = get_device()
print(f"device: {device}")
if not CHECKPOINT.exists():
    raise SystemExit(f"Missing {CHECKPOINT} -- run 05_train_world_model.ipynb first")

In [ ]:
enc, rssm_cfg, dec, heads = cfg["encoder"], cfg["rssm"], cfg.get("decoder", {}), cfg.get("heads", {})
model = WorldModel.from_config_dims(
    embed_dim=int(enc["embed_dim"]),
    encoder_channels=tuple(int(c) for c in enc["channels"]),
    action_dim=int(cfg["env"]["action_dim"]),
    deter_dim=int(rssm_cfg["deter_dim"]),
    stoch=int(rssm_cfg["stoch"]),
    classes=int(rssm_cfg["classes"]),
    hidden=int(rssm_cfg["hidden"]),
    unimix=float(rssm_cfg.get("unimix", 0.01)),
    act=str(rssm_cfg.get("act", "silu")),
    initial=str(rssm_cfg.get("initial", "learned")),
    rec_depth=int(rssm_cfg.get("rec_depth", 1)),
    decoder_channels=tuple(int(c) for c in dec.get("channels", [512, 256, 128, 64])),
    head_hidden=int(heads.get("hidden", 512)),
    head_layers=int(heads.get("layers", 2)),
).to(device)
ckpt = torch.load(CHECKPOINT, weights_only=False, map_location=device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
print(f"loaded {CHECKPOINT} at step {ckpt.get('step')}")

buffer = ReplayBuffer(seed=0)
buffer.load_state_dict(torch.load(cfg["collect"]["out_path"], weights_only=False))
print(f"replay: episodes={len(buffer)} steps={buffer.num_steps}")

In [ ]:
# --- Cache frozen [h,z] features + matching real frames, once. ---
# This is the only step that runs the (expensive) encoder + RSSM. Everything
# after this trains a fresh decoder against a plain in-memory tensor cache, so
# it's much faster per-step than full joint training was.
N_SEQ = 300
SEQ_LEN = 32

feat_cache: list[torch.Tensor] = []
obs_cache: list[torch.Tensor] = []
with torch.no_grad():
    for _ in range(N_SEQ):
        batch = buffer.sample(8, SEQ_LEN)
        obs = batch["obs"].to(device)
        act = one_hot_action(batch["actions"].to(device), model.rssm.action_dim)
        embeds = model.encode(obs)
        rssm_out = model.rssm.observe(embeds, act)
        feat = rssm_features(rssm_out.h, rssm_out.z_posterior)
        feat_cache.append(feat.reshape(-1, feat.shape[-1]).cpu())
        obs_cache.append(obs.reshape(-1, *obs.shape[2:]).cpu())

feat_all = torch.cat(feat_cache, dim=0)
obs_all = torch.cat(obs_cache, dim=0)
n_total = feat_all.shape[0]
n_eval = max(1, n_total // 20)
perm = torch.randperm(n_total)
eval_idx, train_idx = perm[:n_eval], perm[n_eval:]
print(f"cached {n_total} frames (feat_dim={feat_all.shape[-1]}): {len(train_idx)} train / {len(eval_idx)} eval")

In [ ]:
# --- Train a fresh decoder from scratch on the cached, frozen features. ---
probe_decoder = Decoder(embed_dim=model.feat_dim, channels=tuple(int(c) for c in dec.get("channels", [512, 256, 128, 64]))).to(device)
optim = torch.optim.Adam(probe_decoder.parameters(), lr=3e-4)

RECON_SCALE = float(cfg["train"]["recon_scale"])
GRAD_SCALE = float(cfg["train"].get("grad_scale", 1.5))
STEPS = 4000
BATCH = 64
LOG_EVERY = 100

history: list[dict] = []


def eval_loss() -> float:
    probe_decoder.eval()
    with torch.no_grad():
        f = feat_all[eval_idx].to(device)
        o = nhwc_uint8_to_nchw_float(obs_all[eval_idx].to(device))
        pred = probe_decoder(f)
        loss = F.l1_loss(pred, o).item()
    probe_decoder.train()
    return loss


def show_progress() -> None:
    clear_output(wait=True)
    xs = [h["step"] for h in history]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(xs, [h["train_recon"] for h in history], label="train recon (L1)")
    ax.plot(xs, [h["eval_recon"] for h in history], label="eval recon (L1)")
    ax.axhline(orig_recon_on_cache, color="gray", ls="--", label="original decoder (same features)")
    ax.set_title("probe decoder: recon loss on frozen [h,z] features")
    ax.legend(fontsize=8)
    display(fig)
    plt.close(fig)
    print(f"step {history[-1]['step']}/{STEPS}  train={history[-1]['train_recon']:.4f}  eval={history[-1]['eval_recon']:.4f}  (original={orig_recon_on_cache:.4f})")


# Baseline: how well did the ORIGINAL jointly-trained decoder do on this same
# cached feature set? This is the number the probe decoder needs to beat.
with torch.no_grad():
    f = feat_all[eval_idx].to(device)
    o = nhwc_uint8_to_nchw_float(obs_all[eval_idx].to(device))
    orig_recon_on_cache = F.l1_loss(model.decoder(f), o).item()
print(f"original decoder eval L1 on cached features: {orig_recon_on_cache:.4f}")

t0 = time.time()
for step in range(1, STEPS + 1):
    idx = train_idx[torch.randint(0, len(train_idx), (BATCH,))]
    f = feat_all[idx].to(device)
    o = nhwc_uint8_to_nchw_float(obs_all[idx].to(device))
    pred = probe_decoder(f)
    recon = F.l1_loss(pred, o)
    grad = gradient_l1_loss(pred, o)
    loss = RECON_SCALE * recon + GRAD_SCALE * grad
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()

    if step % LOG_EVERY == 0 or step == 1:
        history.append({"step": step, "train_recon": float(recon), "eval_recon": eval_loss()})
        show_progress()

print(f"done in {time.time() - t0:.1f}s")

## Verdict: side-by-side comparison

`real | original decoder | probe decoder`, all decoding the exact same frozen
`[h,z]` features. If the probe column shows clearer mobs/trees/HUD digits than
the middle column, the original decoder was undertrained -- not the latent.

In [ ]:
probe_decoder.eval()
n_show = 6
show_idx = eval_idx[:n_show]
with torch.no_grad():
    f = feat_all[show_idx].to(device)
    real = obs_all[show_idx]
    orig_pred = nchw_float_to_nhwc_uint8(model.decoder(f).cpu())
    probe_pred = nchw_float_to_nhwc_uint8(probe_decoder(f).cpu())

strips = [
    np.concatenate([real[i].numpy(), orig_pred[i].numpy(), probe_pred[i].numpy()], axis=1)
    for i in range(n_show)
]
grid = np.concatenate(strips, axis=0)
fig, ax = plt.subplots(figsize=(6, 2 * n_show))
ax.imshow(grid)
ax.axis("off")
ax.set_title("real | original decoder | probe decoder")
plt.tight_layout()
plt.show()

final_eval = eval_loss()
improvement = 1.0 - final_eval / max(orig_recon_on_cache, 1e-8)
print(f"original decoder eval L1: {orig_recon_on_cache:.4f}")
print(f"probe decoder eval L1:    {final_eval:.4f}  ({improvement:+.0%} vs original)")
print()
if improvement > 0.15:
    print(
        "VERDICT: probe decoder meaningfully beats the original on the SAME frozen "
        "features -> the latent had the information; the original decoder was "
        "undertrained/starved by competing losses, not blind. Fix: give the decoder "
        "more dedicated training (e.g. a decoder-only fine-tune pass), don't touch "
        "the RSSM/encoder/KL balance."
    )
else:
    print(
        "VERDICT: probe decoder does NOT meaningfully beat the original even with "
        "full dedicated attention -> the [h,z] latent itself likely isn't encoding "
        "this content. This is a real RSSM/encoder-side limitation worth knowing "
        "about before M4 (imagined rollouts rely on the same latent)."
    )